# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [3]:
# imports
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display

load_dotenv(override=True)

openai_client = OpenAI()


In [8]:
# Load environment variables in a file called .env

load_dotenv(override=True)

True

In [9]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [10]:
# set up environment
api_key = os.getenv("OPENAI_API_KEY")

# Check the key
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [11]:
# here is the question; type over this to ask something new
#yield from {book.get("author") for book in books if book.get("author")}

question = """
Please explain what this code does and why:
async with AsyncExitStack() as stack:
            trader_mcp_servers = [
                await stack.enter_async_context(
                    MCPServerStdio(params, client_session_timeout_seconds=120)
                )
                for params in trader_mcp_server_params
            ]
"""

In [12]:
# Get gpt-4o-mini to answer, with streaming
system_prompt = """
You are a helpful assistant that can answer questions about code.
You are given a question and a code snippet.
You need to answer the question based on the code snippet.
"""

stream = openai_client.chat.completions.create(
    model=MODEL_GPT,
    messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": question}],
    stream=True
)
response = ""
display_handle = display(Markdown(""), display_id=True)
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)



The provided code snippet is using asynchronous context management to create and manage multiple instances of `MCPServerStdio` (presumably a server-related class) in a clean and efficient manner. Let's break it down step-by-step:

1. **AsyncExitStack**:
   - This is part of the `contextlib` library and provides a way to manage multiple context managers (e.g., resources that need to be cleaned up) in an asynchronous context. It allows you to enter multiple asynchronous context managers and ensures that they are properly exited, even if an error occurs.

2. **`async with`**:
   - This is an asynchronous context manager that allows you to use the `AsyncExitStack`. When the block of code is finished, whether normally or via an exception, the context stack will ensure that all entered contexts are exited properly.

3. **List Comprehension**:
   - Inside the `async with` block, there is a list comprehension that creates a list called `trader_mcp_servers`. For each set of parameters in `trader_mcp_server_params`, it does the following:
     - Calls `await stack.enter_async_context(...)`, which enters an asynchronous context for the `MCPServerStdio` instance. This means that each instance of `MCPServerStdio` is awaited to ensure it has been properly initialized and is ready to be used asynchronously.
     - `MCPServerStdio(params, client_session_timeout_seconds=120)` is being instantiated with the provided parameters and with a `client_session_timeout_seconds` argument set to 120 seconds.

4. **Overall Purpose**:
   - The overall purpose of this code snippet is to create multiple instances of `MCPServerStdio` using the parameters defined in `trader_mcp_server_params`. Each instance will be managed by the `AsyncExitStack`, which takes care of proper resource management—ensuring that when the context is exited (whether normally or due to an error), all resources (like sockets, files, etc.) associated with the `MCPServerStdio` instances are cleaned up correctly.

### Summary
This code asynchronously initializes a list of `MCPServerStdio` instances while ensuring that each instance is properly managed (entered and exited) in a clean and safe manner using `AsyncExitStack`. This is particularly useful when dealing with multiple resource-heavy operations where proper cleanup is critical to prevent resource leaks or other issues.

In [ ]:
# Get Llama 3.2 to answer